In [4]:
import pandas as pd
import geopandas as gpd
import numpy as np
import ast

In [5]:
visum_links = gpd.read_file(r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/Insumos OSMNX/Visum/Edges/edges_visum.shp')

In [4]:
visum_links['highway'].apply(type).value_counts()

highway
<class 'str'>         492462
<class 'NoneType'>      2434
Name: count, dtype: int64

## __Normalize highway column__

In [7]:
def parse_highway(val):
    if isinstance(val, str) and val.startswith('['):
        try:
            return ast.literal_eval(val)
        except Exception:
            return [val]
    elif isinstance(val, str):
        return [val]
    elif isinstance(val, list):
        return val
    else:
        return []

In [8]:
visum_links['highway_norm'] = visum_links['highway'].apply(parse_highway)

## __Assign Modes column__

In [17]:
# Tsys = WALK
walk_types = {
    'unclassified', 'residential', 'service',
    'footway', 'path', 'living_street', 'pedestrian', 'track', 'steps',
    'corridor', 'bridleway', 'ladder'
}

#Tsys = CAR, BUS
car_types = {
    'motorway', 'motorway_link', 
    'trunk', 'trunk_link', 
    'primary', 'primary_link', 
    'secondary', 'secondary_link',
    'tertiary', 'tertiary_link',
    'unclassified', 'residential', 'service',
}

# Tsys = BIKE
bicycle_types = {'cycleway'}

# Tsys = BRT
BRT_types = {'busway'}

# Tsys = TREN LIGERO
tren_ligero_names = {
    'Línea 1 del Tren Eléctrico Urbano',
    'Linea 1 del Tren Eléctrico Urbano', 
    'Línea 2 del Tren Eléctrico Urbano',
    'Linea 2 del Tren Eléctrico Urbano',
    'Línea 3 del Tren Eléctrico Urbano',
    'Linea 3 del Tren Eléctrico Urbano',
    'Línea 4 del Tren Eléctrico Urbano de Guadalajara',
    'Linea 4 del Tren Eléctrico Urbano de Guadalajara'
}

In [44]:
def assign_mode(row):
    highway = row.get('highway_norm', [])
    name = row.get('name', '')

    # if highway is empty
    if not highway and name in tren_ligero_names:
        return 'TREN LIGERO'
    
    #Exclusive link BRT
    if set(highway) & BRT_types:
        return 'BRT'

    modes = []
    if set(highway) & bicycle_types:
        modes.append('BIKE')
    if set(highway) & car_types:
        modes.append('BUS')
        modes.append('CAR')
    if set(highway) & walk_types:
        modes.append('WALK')

    #return modes
    return ",".join(sorted(set(modes)))

In [45]:
visum_links['Modes'] = visum_links.apply(assign_mode, axis=1)

In [46]:
mode_to_tsys = {
    'CAR':'C',
    'BUS':'B',
    'WALK':'W',
    'BIKE':'Bicycle',
    'BRT':'BRT',
    'TREN LIGERO':'TL',
}

def define_tsys(row):
    modes_str = row.get('Modes', '')
    modes = [m.strip() for m in modes_str.split(',')]
    tsys = [mode_to_tsys[m] for m in modes if m in mode_to_tsys]
    tsys_str = ",".join(tsys)
    return tsys_str

In [47]:
visum_links['TsysCodes'] = visum_links.apply(define_tsys, axis=1)

In [7]:
visum_links['length_km'] = visum_links['length'] / 1000
visum_links

,u,v,key,osmid,highway,lanes,name,oneway,ref,reversed,...,LinkNo,AllowedMod,Length_m,OneWay_1,Name_1,highway_no,Modes,TsysCodes,geometry,length_km
0,267537966,7306651630,0,"[832652768, 619339782, 688424559, 188974544, 1...",motorway,"['2', '3']",Autopista Guadalajara - Morelia,True,MEX 15D;MEX 80D,False,...,1,"B,C",4832.636188,1,Autopista Guadalajara - Morelia,['motorway'],"BUS,CAR","B,C","LINESTRING (-103.24632 20.61606, -103.24734 20...",4.827825
1,267537966,5837556433,0,694566994,motorway_link,1,None,True,None,False,...,2,"B,C",146.226523,1,None,['motorway_link'],"BUS,CAR","B,C","LINESTRING (-103.24632 20.61606, -103.24648 20...",0.146029
2,267538751,273140976,0,189118222,motorway,2,Autopista Guadalajara - Zapotlanejo,True,MEX 80D;MEX 90D,False,...,3,"B,C",520.548789,1,Autopista Guadalajara - Zapotlanejo,['motorway'],"BUS,CAR","B,C","LINESTRING (-103.1364 20.60565, -103.13432 20....",0.520237
3,267538751,1746293763,0,907206852,motorway_link,1,Autopista Guadalajara - Morelia,True,MEX 15D,False,...,4,"B,C",617.879708,1,Autopista Guadalajara - Morelia,['motorway_link'],"BUS,CAR","B,C","LINESTRING (-103.1364 20.60565, -103.1354 20.6...",0.617213
4,273140976,1997658287,0,835083267,motorway,3,Autopista Guadalajara - Zapotlanejo,True,MEX 80D;MEX 90D,False,...,5,"B,C",151.478387,1,Autopista Guadalajara - Zapotlanejo,['motorway'],"BUS,CAR","B,C","LINESTRING (-103.13185 20.60758, -103.13162 20...",0.151355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494891,13656279161,5538433659,0,577828524,None,None,Linea 3 del Tren Eléctrico Urbano,False,None,False,...,494892,{'TL'},17.719509,0,Linea 3 del Tren Eléctrico Urbano,[],TREN LIGERO,TL,"LINESTRING (-103.39072 20.7291, -103.39086 20....",0.017725
494892,13656279161,5538433658,0,577828524,None,None,Linea 3 del Tren Eléctrico Urbano,False,None,True,...,494893,{'TL'},20.002953,0,Linea 3 del Tren Eléctrico Urbano,[],TREN LIGERO,TL,"LINESTRING (-103.39072 20.7291, -103.39055 20....",0.019997
494893,13705424642,4591349866,0,463965070,None,None,Línea 2 del Tren Eléctrico Urbano,False,None,False,...,494894,{'TL'},49.828305,0,Línea 2 del Tren Eléctrico Urbano,[],TREN LIGERO,TL,"LINESTRING (-103.3559 20.6749, -103.35637 20.6...",0.049754
494894,13705424642,4591553254,0,463965070,None,None,Línea 2 del Tren Eléctrico Urbano,False,None,True,...,494895,{'TL'},1.844162,0,Línea 2 del Tren Eléctrico Urbano,[],TREN LIGERO,TL,"LINESTRING (-103.3559 20.6749, -103.35588 20.6...",0.001841


In [8]:
visum_links.to_file(r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/Insumos OSMNX/Visum/Edges/edges_visum.shp')

In [30]:
visum_links[visum_links['highway'].apply(lambda x: isinstance(x, list))]

,u,v,key,osmid,highway,lanes,name,oneway,ref,reversed,...,FromNodeNo,ToNodeNo,LinkNo,AllowedMod,Length_m,OneWay_1,Name_1,geometry,highway_norm,Modes


In [50]:
visum_links[visum_links['TsysCodes']=='TL']

,u,v,key,osmid,highway,lanes,name,oneway,ref,reversed,...,ToNodeNo,LinkNo,AllowedMod,Length_m,OneWay_1,Name_1,geometry,highway_norm,Modes,TsysCodes
492462,294101514,1377754026,0,463705537,None,None,Línea 1 del Tren Eléctrico Urbano,False,None,False,...,205271,492463,{'TL'},143.789698,0,Línea 1 del Tren Eléctrico Urbano,"LINESTRING (-103.35589 20.71091, -103.35615 20...",[],TREN LIGERO,TL
492463,294101514,1377754010,0,463705537,None,None,Línea 1 del Tren Eléctrico Urbano,False,None,True,...,205270,492464,{'TL'},14.984365,0,Línea 1 del Tren Eléctrico Urbano,"LINESTRING (-103.35589 20.71091, -103.35585 20...",[],TREN LIGERO,TL
492464,294101519,294103108,0,463705537,None,None,Línea 1 del Tren Eléctrico Urbano,False,None,False,...,205209,492465,{'TL'},48.745238,0,Línea 1 del Tren Eléctrico Urbano,"LINESTRING (-103.3548 20.70609, -103.3548 20.7...",[],TREN LIGERO,TL
492465,294101519,1377754093,0,463705537,None,None,Línea 1 del Tren Eléctrico Urbano,False,None,True,...,205273,492466,{'TL'},35.766970,0,Línea 1 del Tren Eléctrico Urbano,"LINESTRING (-103.3548 20.70609, -103.35486 20....",[],TREN LIGERO,TL
492466,294101528,9499989139,0,463705537,None,None,Línea 1 del Tren Eléctrico Urbano,False,None,False,...,205650,492467,{'TL'},185.826603,0,Línea 1 del Tren Eléctrico Urbano,"LINESTRING (-103.35401 20.69315, -103.35395 20...",[],TREN LIGERO,TL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494891,13656279161,5538433659,0,577828524,None,None,Linea 3 del Tren Eléctrico Urbano,False,None,False,...,206205,494892,{'TL'},17.719509,0,Linea 3 del Tren Eléctrico Urbano,"LINESTRING (-103.39072 20.7291, -103.39086 20....",[],TREN LIGERO,TL
494892,13656279161,5538433658,0,577828524,None,None,Linea 3 del Tren Eléctrico Urbano,False,None,True,...,206204,494893,{'TL'},20.002953,0,Linea 3 del Tren Eléctrico Urbano,"LINESTRING (-103.39072 20.7291, -103.39055 20....",[],TREN LIGERO,TL
494893,13705424642,4591349866,0,463965070,None,None,Línea 2 del Tren Eléctrico Urbano,False,None,False,...,206097,494894,{'TL'},49.828305,0,Línea 2 del Tren Eléctrico Urbano,"LINESTRING (-103.3559 20.6749, -103.35637 20.6...",[],TREN LIGERO,TL
494894,13705424642,4591553254,0,463965070,None,None,Línea 2 del Tren Eléctrico Urbano,False,None,True,...,206098,494895,{'TL'},1.844162,0,Línea 2 del Tren Eléctrico Urbano,"LINESTRING (-103.3559 20.6749, -103.35588 20.6...",[],TREN LIGERO,TL
